<a href="https://colab.research.google.com/github/aahmedd38/Enhance-labeling-model/blob/main/RAG_GRC_Consultant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
pydantic>=2.0.0 \
openai>=1.0.0 \
lancedb==0.22.0 \
docling==2.31.0 \
cohere==5.15.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.15.4 which is incompatible.


In [ ]:
import pydantic
import openai
import lancedb
import docling
import cohere

print("All installed successfully")

All installed successfully


Load the Docs + check the doc's and count number of pages



In [9]:
import os
from pypdf import PdfReader

folder_path = "docs"
documents = []
for file_name in os.listdir(folder_path):
    if not file_name.endswith(".pdf"):
        continue
    file_path = os.path.join(folder_path, file_name)
    reader = PdfReader(file_path)
    print(f"Doc: {file_name} ({len(reader.pages)} pages)")

    for page_num, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text and page_text.strip():
            documents.append({
                "source": file_name,
                "page": page_num + 1,
                "text": page_text.strip()
            })

print(f"\n Documents : {len(documents)}")

Doc: ISO-27001.pdf (26 pages)
Doc: Compliance_Management.pdf (56 pages)
Doc: PCI-DSS.pdf (397 pages)
Doc: Frameworks_ISO27_Guideline.pdf (88 pages)
Doc: Compliance_Management_2.pdf (114 pages)
Doc: Governance_Startegy.pdf (41 pages)
Doc: Risk_Assessment_Management.pdf (56 pages)

 Documents : 778


chunking the data , each chunk is 500 characters

-> here use overlap 100 , to keep the context of the data [the new chunk will contain last 100 chars of the previous chunk]

first for each doc put a metadata that tell us what is that

> ***-> then use the spiltter(function) to chunk data across [500 chars + overlab 100 + split over paragragh , then line then sentence then spaces] ***





In [12]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\.{3,}', '', text)
    text = re.sub(r'\b\d+\b(?=\s*$)', '', text)
    return text.strip()

doc_metadata = {
    "PCI-DSS.pdf":                    "payment_security",
    "Compliance_Management.pdf":      "compliance",
    "Compliance_Management_2.pdf":    "compliance",
    "Frameworks_ISO27_Guideline":      "iso27000",
    "ISO_27001.pdf":                  "iso27000",
    "Risk_Assessment_Management.pdf": "risk",
    "Governance_Startegy.pdf":        "governance"
}
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

def chunk_documents(documents):
    chunked_docs= []
    for doc in documents:
        cleaned = clean_text(doc["text"])
        chunks= splitter.split_text(cleaned)
        for i, chunk in enumerate(chunks):
            if len(chunk.strip()) > 50:
                chunked_docs.append({
                    "source":   doc["source"],
                    "page":     doc["page"],
                    "chunk_id": i,
                    "domain":   doc_metadata.get(doc["source"], "general"),
                    "text":     chunk.strip()
                })
    return chunked_docs
def unique(chunked_docs):
    seen = set()
    uniq = []
    for chunk in chunked_docs:
        key = re.sub(r'\s+', ' ', chunk["text"].lower().strip())[:200]
        if key not in seen:
            seen.add(key)
            uniq.append(chunk)
    return uniq
chunked_docs = chunk_documents(documents)
chunked_docs = unique(chunked_docs)

print(f"Total chunks:{len(chunked_docs)}")

Total chunks:2528


Embedding step :


> first :
  1- load the model
  2- load all text


> embedding each teaxt and convert each text to vector :
1- take the text
2- encode 64 in one time to make it faster
3- return numpy array that for the FAISS model
-------------------------------------------
Vector_DB (FAISS) :
> embeddings.shape=(number_of_vectors,size):
   ##embeddings.shape[1] -> will takes sizeof vectors as the dimension
*   
> assign the dimention as the size of vectors
-> then search engine structure that used Eculidian distance formula between the vectors , then add the vectors to the structure

##Now have vectors with 384-dimention of [data stored] ready for search on it and find distance between [user query & data stored]







In [16]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model =SentenceTransformer("all-MiniLM-L6-v2")
texts=[doc["text"] for doc in chunked_docs]
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
dimension = embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(embeddings)

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

# Semantic Search [between user_query & Vector_DB data]



> take the user query and 5 is related for best 5 related chunks
- embedding user query and convert it to numpy array to pass it to the FAISS Vector_DB
- will compare user_embedded_query with all the DB Vectors
- will return 2 arrays : #distance[n1,n2,...] from best to worst --- ##slots:slots[s1,s2,...] address for each vector on the FAISS_Vector_DB
- ok after searching , now map each founded slot with their chunks of the docs  


In [17]:
import numpy as np
def search(user_query, k=5):
    query_embedding = model.encode([user_query], convert_to_numpy=True).astype("float32")
    distances, slots = index.search(query_embedding, k)
    results = []
    for i, idx in enumerate(slots[0]):
        chunk=chunked_docs[idx].copy()
        chunk["distance"] =round(float(distances[0][i]), 4)
        results.append(chunk)
    return results

if iwant can add : - End with: Source: <filename> | Page <number>

In [29]:
from openai import OpenAI

llm=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="KEY",
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title":      "GRC Assistant"
    }
)

def build_context(results):
    context = ""
    for i, r in enumerate(results):
        context += f"""
[{i+1}]
Source : {r['source']}
Page   : {r['page']}
Domain : {r['domain']}
Text   : {r['text']}
---"""
    return context

def rag(question, k=5):
    results = search(question,k=5)
    results = [r for r in results if r["distance"] < 1.0]

    context = build_context(results)

    prompt = f"""

You are a senior cybersecurity GRC consultant.
Answer the question in a clear, professional, and structured paragraph format.
Use ONLY information from context.
If repeated information appears, merge it.
Do NOT repeat same source multiple times.
Always format citations uniquely.

Rules:
- Be accurate and professional
- Reference frameworks and controls when possible
- If information is missing, say so clearly
- Do not hallucinate
- Keep answers concise and avoid unnecessary repetition or long academic explanations.

Context:
{context}

Question: {question}

Answer:"""
    response = llm.chat.completions.create(
        model="openrouter/free",
        messages=[
            {"role": "system", "content": "You are an senior cybersecurity GRC consultant. Answer directly and concisely."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.2,
        max_tokens=1000
    )
    message = response.choices[0].message
    if message.content and len(message.content.strip()) > 5:
        return message.content.strip()
    return "Model returned empty response. Try again."

#test
questions = [
    "What are PCI DSS password requirements?",
    "write a policy for installing ant-malwares on employees devices",
    "What are information security controls?"
]

for q in questions:
    print("=" * 60)
    print(f"{q}")
    print(f"{rag(q)}")
    print()

What are PCI DSS password requirements?
PCI DSS password requirements include a minimum password length of seven characters until 31 March 2025 per Requirement 8.2.3, and the use of strong cryptography for all non-console administrative access (Requirement 2.2.7) and for protecting passwords in transit and storage [1, 4]. All cryptographic cipher suites and protocols used to meet PCI DSS requirements—including those protecting passwords and authenticating access—must comply with this standard, which is a best practice until 31 March 2025 and then becomes mandatory [2]. Default passwords on system administration, user, or service accounts must be changed or removed, as they are well known and easily guessed [5]. Importantly, other password controls such as intruder lockout or complex passwords cannot compensate for the absence of encrypted passwords, since they do not mitigate the risk of interception of cleartext passwords [3]. Additional requirements apply where insecure services or p